# 4. The STRING gene graph

Question: which pairs of candidate genes does STRING connect, so that notebook 5 can build a graph on any selected gene list?

What enters: the candidate gene pool of notebook 1 and two STRING files, the protein links with their combined score and the protein names.

What leaves: one gene-level edge list at combined score 700 or more in `results/04_gene_graph/`, with its source table and a run record.

Result: the line under the imports is rendered from the saved summary table.

How to run: `uv run jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.record_timing=False notebooks/04_gene_graph.ipynb`; the first run downloads about 65 MB.

In [1]:
import hashlib
import json
import subprocess
import sys
from datetime import date
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from IPython.display import Markdown, display
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

In [2]:
summary_file = Path("../results/04_gene_graph/graph_summary.csv")
if summary_file.exists():
    saved = pd.read_csv(summary_file, index_col=0).loc["score >= 700"]
    display(Markdown(f"Result: {saved['edges']} edges among {saved['genes with an edge']} of the "
                     f"{saved['pool genes']} pool genes at combined score 700 or more."))
else:
    display(Markdown("Result: not yet run."))

Result: 257653.0 edges among 16269.0 of the 19211.0 pool genes at combined score 700 or more.

## Settings

STRING scores a protein pair from 0 to 1000; 400, 700 and 900 are its medium, high and highest confidence cut-offs. The graph model uses 700, the cut-off named in the thesis text.

In [3]:
STRING_VERSION = "12.5"  # the current STRING release on the retrieval date
TAXON = 9606             # Homo sapiens
BASE = "https://stringdb-downloads.org/download"
FILES = {"links": f"protein.links.v{STRING_VERSION}/{TAXON}.protein.links.v{STRING_VERSION}.txt.gz",
         "info": f"protein.info.v{STRING_VERSION}/{TAXON}.protein.info.v{STRING_VERSION}.txt.gz"}
SCORES = [400, 700, 900]  # medium, high and highest confidence
SAVED_SCORE = 700         # the cut-off notebook 5 uses
DATA = Path("../data/raw/string")
RESULTS = Path("../results/04_gene_graph")

## Downloading with a record

Each file is downloaded once and its SHA-256 and retrieval date are written beside it; a second run downloads nothing. The source table records release, address, licence and citation.

In [4]:
def fetch(name):
    """Download one STRING file unless it is on disk, and return its SHA-256."""
    path = DATA / Path(FILES[name]).name
    if not path.exists():
        answer = requests.get(f"{BASE}/{FILES[name]}", timeout=600)
        answer.raise_for_status()
        path.write_bytes(answer.content)
    return hashlib.sha256(path.read_bytes()).hexdigest()


DATA.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(exist_ok=True)
manifest_file = DATA / "manifest.json"
if not manifest_file.exists():
    hashes = {}
    for name in FILES:
        hashes[name] = fetch(name)
    manifest = {"retrieved_on": date.today().isoformat(), "sha256": hashes}
    manifest_file.write_text(json.dumps(manifest, indent=1))
manifest = json.loads(manifest_file.read_text())

In [5]:
rows = []
for name, part in FILES.items():
    row = {"file": Path(part).name, "release": f"STRING {STRING_VERSION}",
           "where from": f"{BASE}/{part}", "retrieved on": manifest["retrieved_on"],
           "sha256": manifest["sha256"][name],
           "licence": "Creative Commons BY 4.0, as stated at string-db.org/cgi/access",
           "citation": "Szklarczyk et al., Nucleic Acids Res 2025, 53:D730, "
                       "doi:10.1093/nar/gkae1113"}
    rows.append(row)
sources = pd.DataFrame(rows).set_index("file")
sources.to_csv(RESULTS / "sources.csv")
sources.T

file,9606.protein.links.v12.5.txt.gz,9606.protein.info.v12.5.txt.gz
release,STRING 12.5,STRING 12.5
where from,https://stringdb-downloads.org/download/protei...,https://stringdb-downloads.org/download/protei...
retrieved on,2026-10-29,2026-10-29
sha256,3d6894ef3a0a8578b70421b7d6bac159c41ed2f6c1b5da...,144de4b0d98c6a7dfde6ddc2591cf88657f27b989eadff...
licence,"Creative Commons BY 4.0, as stated at string-d...","Creative Commons BY 4.0, as stated at string-d..."
citation,"Szklarczyk et al., Nucleic Acids Res 2025, 53:...","Szklarczyk et al., Nucleic Acids Res 2025, 53:..."


## Proteins to pool genes

STRING names proteins; the pool names genes by Ensembl ID with a symbol. A protein maps to the pool gene with the same preferred name; a protein whose name is not in the pool, or a name shared by several proteins, is dropped and counted.

In [6]:
pool = pd.read_csv("../results/01_data/gene_pool.csv")
assert list(pool.columns) == ["gene_id", "symbol", "biotype"], "the gene pool carries no outcome"
assert pool["symbol"].is_unique, "a symbol names two pool genes"
info_file = DATA / Path(FILES["info"]).name
info = pd.read_csv(info_file, sep="\t", usecols=["#string_protein_id", "preferred_name"])
in_pool = info[info["preferred_name"].isin(pool["symbol"])]
shared_name = in_pool["preferred_name"].duplicated(keep=False)
mapping = in_pool[~shared_name].merge(pool, left_on="preferred_name", right_on="symbol")
protein_to_gene = mapping.set_index("#string_protein_id")["gene_id"]
display(Markdown(f"{len(info)} human proteins; {len(in_pool)} carry a pool symbol; "
                 f"{int(shared_name.sum())} share their symbol with another protein and are "
                 f"dropped; {protein_to_gene.size} proteins map to one pool gene each."))

19699 human proteins; 18007 carry a pool symbol; 0 share their symbol with another protein and are dropped; 18007 proteins map to one pool gene each.

## Protein links to gene edges

STRING lists every pair in both directions. We keep the pairs whose two proteins map to pool genes, make them undirected, drop self-pairs and keep the highest score per gene pair.

In [7]:
links = pd.read_csv(DATA / Path(FILES["links"]).name, sep=" ")
links["gene_a"] = links["protein1"].map(protein_to_gene)
links["gene_b"] = links["protein2"].map(protein_to_gene)
mapped = links.dropna(subset=["gene_a", "gene_b"])
first = mapped["gene_a"].to_numpy()
second = mapped["gene_b"].to_numpy()
ordered = {"gene_a": np.where(first < second, first, second),
           "gene_b": np.where(first < second, second, first),
           "score": mapped["combined_score"].to_numpy()}
pairs = pd.DataFrame(ordered)
pairs = pairs[pairs["gene_a"] != pairs["gene_b"]]
edges = pairs.groupby(["gene_a", "gene_b"], as_index=False)["score"].max()
display(Markdown(f"{len(links)} directed protein links; {len(mapped)} between pool genes; "
                 f"{len(edges)} undirected gene pairs."))

10826054 directed protein links; 9928734 between pool genes; 4964367 undirected gene pairs.

## The graph at three cut-offs

The table counts the graph at each cut-off; the degree of a gene is its number of edges, and a connected component is a set of genes linked by some path.

In [8]:
def graph_counts(edge_table, score):
    """Edges, genes with an edge, mean degree and largest component at one cut-off."""
    kept = edge_table[edge_table["score"] >= score]
    genes = pd.Index(pd.unique(kept[["gene_a", "gene_b"]].to_numpy().ravel()))
    weights = np.ones(len(kept))
    position = (genes.get_indexer(kept["gene_a"]), genes.get_indexer(kept["gene_b"]))
    adjacency = coo_matrix((weights, position), shape=(len(genes), len(genes)))
    components, labels = connected_components(adjacency, directed=False)
    return {"edges": len(kept), "genes with an edge": len(genes),
            "mean degree": round(2 * len(kept) / len(genes), 1),
            "largest connected component (genes)": int(np.bincount(labels).max())}


summary_rows = {}
for score in SCORES:
    summary_rows[f"score >= {score}"] = graph_counts(edges, score)
summary = pd.DataFrame.from_dict(summary_rows, orient="index")
summary.insert(0, "pool genes", len(pool))
summary.insert(1, "pool genes with a STRING protein", protein_to_gene.nunique())
summary.to_csv(RESULTS / "graph_summary.csv")
summary

,pool genes,pool genes with a STRING protein,edges,genes with an edge,mean degree,largest connected component (genes)
score >= 400,19211,18007,994865,17986,110.6,17986
score >= 700,19211,18007,257653,16269,31.7,16158
score >= 900,19211,18007,113848,13339,17.1,12800


In [9]:
at_700 = summary.loc[f"score >= {SAVED_SCORE}"]
display(Markdown(f"At score {SAVED_SCORE} or more, {at_700['genes with an edge']} of the "
                 f"{len(pool)} pool genes have at least one edge, with a mean degree of "
                 f"{at_700['mean degree']}; a selected gene without an edge stays an isolated "
                 "node in notebook 5."))

At score 700 or more, 16269.0 of the 19211 pool genes have at least one edge, with a mean degree of 31.7; a selected gene without an edge stays an isolated node in notebook 5.

## The saved edge list

The edges at 700 or more are saved as a gzip-compressed table of two Ensembl IDs and the score, which notebook 5 filters to each selected gene list. No patient data enters this file.

In [10]:
saved_edges = edges[edges["score"] >= SAVED_SCORE].sort_values(["gene_a", "gene_b"])
assert saved_edges["gene_a"].isin(pool["gene_id"]).all(), "a node is not a pool gene"
assert saved_edges["gene_b"].isin(pool["gene_id"]).all(), "a node is not a pool gene"
assert (saved_edges["gene_a"] != saved_edges["gene_b"]).all(), "a self-loop"
assert not saved_edges.duplicated(["gene_a", "gene_b"]).any(), "a duplicate pair"
edge_file = RESULTS / f"edges_{SAVED_SCORE}.csv.gz"
gzip_without_timestamp = {"method": "gzip", "mtime": 0}  # a fixed timestamp keeps the hash stable
saved_edges.to_csv(edge_file, index=False, compression=gzip_without_timestamp)
display(Markdown(f"{len(saved_edges)} edges saved, {edge_file.stat().st_size / 1e6:.1f} MB."))

257653 edges saved, 1.8 MB.

## The earlier pipeline's graph

The earlier pipeline (branch `depricated`, `src/kg_construction.py` and `configs/config.yaml`) kept STRING edges at combined score 300 or more and added co-expression edges computed on all patients, each gene linked to its ten strongest correlates above 0.6, while the thesis text described a cut-off of 700. This notebook keeps STRING edges at 700 or more and adds no edge computed from the patients.

## Run record

The last block records the git commit, the package versions and the SHA-256 of every input and of the saved edge list. A second run reproduces the same hashes.

In [11]:
def git(*arguments):
    """The output of one git command, for the run record."""
    command = ["git", *arguments]
    return subprocess.run(command, capture_output=True, text=True, check=True).stdout.strip()


packages = {}
for name in ["numpy", "pandas", "scipy", "requests"]:
    packages[name] = version(name)
pool_hash = hashlib.sha256(Path("../results/01_data/gene_pool.csv").read_bytes()).hexdigest()
run_record = {"git commit": git("rev-parse", "HEAD"),
              "tracked files changed since that commit":
                  git("status", "--porcelain", "--untracked-files=no").splitlines(),
              "python": sys.version.split()[0], "packages": packages, "seeds": "none",
              "string files (sha256)": manifest["sha256"], "gene pool (sha256)": pool_hash,
              "edge file (sha256)": hashlib.sha256(edge_file.read_bytes()).hexdigest()}
(RESULTS / "run_record.json").write_text(json.dumps(run_record, indent=1))
print(json.dumps(run_record, indent=1))

{
 "git commit": "ed1721a640e27b5257c90567a1fc42d4ce0cc98f",
 "tracked files changed since that commit": [],
 "python": "3.12.14",
 "packages": {
  "numpy": "2.5.3",
  "pandas": "3.0.6",
  "scipy": "1.18.1",
  "requests": "2.34.2"
 },
 "seeds": "none",
 "string files (sha256)": {
  "links": "3d6894ef3a0a8578b70421b7d6bac159c41ed2f6c1b5da21ab7fabb4ac21cd25",
  "info": "144de4b0d98c6a7dfde6ddc2591cf88657f27b989eadff4f501450c3ed1f0f1c"
 },
 "gene pool (sha256)": "ce6e2da0262e3347d75f05f6b8701be92a915aa27ba6e99ab959671da4ea4426",
 "edge file (sha256)": "e8116725e93b7b20c03b5d83759a48541bf17fd51e179ea95894c382fcf5f79b"
}
